# 03 — In Season Feature Engineering

This notebook creates the weekly performance features used by the in season projection layer.

It does not modify the original preseason notebooks or their outputs.

The purpose of this notebook is to convert completed regular season games into team level offensive and defensive performance features that can later be blended with the frozen preseason model.

For a selected target week, the notebook:

- Loads only completed games from weeks before the target week
- Builds team level scoring and margin summaries
- Separates offense and defense
- Adds simple opponent adjusted performance features
- Applies early season shrinkage to reduce overreaction to small samples
- Saves a clean feature table for the weekly team strength notebook

This notebook is intentionally conservative after Week 1. One game should inform the model, not redefine it.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


## Paths and Weekly Settings

This notebook expects the outputs created by:

`02_Weekly_Data_Update.ipynb`

The only value that normally changes week to week is `TARGET_WEEK`.


In [2]:
PROJECT_ROOT = Path("../..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"

SEASON = 2026
TARGET_WEEK = 3
PRIOR_WEEK = TARGET_WEEK - 1

completed_games_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_completed_games.parquet"
)

prior_strength_path = (
    WEEKLY_DATA_DIR
    / f"week_{PRIOR_WEEK:02d}_team_strength.parquet"
)

print("Using completed games:", completed_games_path)
print("Using entering ratings:", prior_strength_path)

Using completed games: ..\..\data\processed\weekly\week_03_completed_games.parquet
Using entering ratings: ..\..\data\processed\weekly\week_02_team_strength.parquet


# Load Completed Games

Only games completed before the target week should exist in this file.

That leakage rule was enforced in `02_Weekly_Data_Update.ipynb`.


In [3]:
completed_games = pd.read_parquet(
    completed_games_path
)

completed_games = completed_games.sort_values(
    ["week", "gameday", "game_id"]
).reset_index(drop=True)

print(
    f"Completed games available before Week {TARGET_WEEK}:",
    len(completed_games)
)

display(
    completed_games[
        [
            "week",
            "gameday",
            "away_team",
            "away_score",
            "home_team",
            "home_score"
        ]
    ]
)


Completed games available before Week 3: 32


,week,gameday,away_team,away_score,home_team,home_score
0,1,2026-09-09,NE,10.0,SEA,13.0
1,1,2026-09-10,SF,27.0,LA,7.0
2,1,2026-09-13,ARI,26.0,LAC,14.0
3,1,2026-09-13,ATL,13.0,PIT,20.0
4,1,2026-09-13,BAL,41.0,IND,23.0
5,1,2026-09-13,BUF,36.0,HOU,31.0
6,1,2026-09-13,CHI,59.0,CAR,37.0
7,1,2026-09-13,CLE,10.0,JAX,34.0
8,1,2026-09-13,DAL,20.0,NYG,28.0
9,1,2026-09-13,GB,22.0,MIN,39.0


In [4]:
update_games = completed_games[
    completed_games["week"] == PRIOR_WEEK
].copy()

update_games = update_games.sort_values(
    ["gameday", "game_id"]
).reset_index(drop=True)

print(
    f"Week {PRIOR_WEEK} games used to update "
    f"Week {TARGET_WEEK} ratings:",
    len(update_games)
)

display(
    update_games[
        [
            "week",
            "gameday",
            "away_team",
            "away_score",
            "home_team",
            "home_score"
        ]
    ]
)

Week 2 games used to update Week 3 ratings: 16


,week,gameday,away_team,away_score,home_team,home_score
0,2,2026-09-17,DET,31.0,BUF,41.0
1,2,2026-09-20,CAR,34.0,ATL,3.0
2,2,2026-09-20,CIN,20.0,HOU,6.0
3,2,2026-09-20,CLE,23.0,TB,19.0
4,2,2026-09-20,GB,20.0,NYJ,17.0
5,2,2026-09-20,IND,30.0,KC,33.0
6,2,2026-09-20,JAX,13.0,DEN,20.0
7,2,2026-09-20,LV,26.0,LAC,14.0
8,2,2026-09-20,MIA,13.0,SF,35.0
9,2,2026-09-20,MIN,9.0,CHI,3.0


# Convert Prior Week Games to Team Game Rows

Only the most recently completed week is used for the new weekly update.

For Week 3:

- Week 2 team strength is the entering prior
- Week 2 game performance is the new evidence
- Week 1 is not counted again because it is already reflected in the Week 2 ratings

Each game becomes two rows:

- one from the home team's perspective
- one from the away team's perspective

In [5]:
home_rows = pd.DataFrame(
    {
        "game_id": update_games["game_id"],
        "week": update_games["week"],
        "team": update_games["home_team"],
        "opponent": update_games["away_team"],
        "is_home": 1,
        "points_for": update_games["home_score"],
        "points_against": update_games["away_score"]
    }
)

away_rows = pd.DataFrame(
    {
        "game_id": update_games["game_id"],
        "week": update_games["week"],
        "team": update_games["away_team"],
        "opponent": update_games["home_team"],
        "is_home": 0,
        "points_for": update_games["away_score"],
        "points_against": update_games["home_score"]
    }
)

team_games = pd.concat(
    [home_rows, away_rows],
    ignore_index=True
)

team_games["point_margin"] = (
    team_games["points_for"]
    - team_games["points_against"]
)

team_games = team_games.sort_values(
    ["team", "game_id"]
).reset_index(drop=True)

print("Team-game rows:", len(team_games))

display(team_games)

Team-game rows: 32


,game_id,week,team,opponent,is_home,points_for,points_against,point_margin
0,2026_02_SEA_ARI,2,ARI,SEA,1,7.0,31.0,-24.0
1,2026_02_CAR_ATL,2,ATL,CAR,1,3.0,34.0,-31.0
2,2026_02_NO_BAL,2,BAL,NO,1,17.0,24.0,-7.0
3,2026_02_DET_BUF,2,BUF,DET,1,41.0,31.0,10.0
4,2026_02_CAR_ATL,2,CAR,ATL,0,34.0,3.0,31.0
5,2026_02_MIN_CHI,2,CHI,MIN,1,3.0,9.0,-6.0
6,2026_02_CIN_HOU,2,CIN,HOU,0,20.0,6.0,14.0
7,2026_02_CLE_TB,2,CLE,TB,0,23.0,19.0,4.0
8,2026_02_WAS_DAL,2,DAL,WAS,1,37.0,20.0,17.0
9,2026_02_JAX_DEN,2,DEN,JAX,1,20.0,13.0,7.0


# Attach Entering Team and Opponent Strength

Each team's performance is evaluated relative to what the model believed before the game was played.

For Week 3 projections, Week 2 ratings represent the information available entering Week 2.

This prevents the opponent adjustment from using future information or the same results that are currently being evaluated.

In [6]:
prior_strength = pd.read_parquet(
    prior_strength_path
).copy()

required_prior_columns = [
    "team",
    "weekly_team_strength",
    "weekly_strength_rank"
]

missing_prior_columns = [
    col
    for col in required_prior_columns
    if col not in prior_strength.columns
]

if missing_prior_columns:
    raise KeyError(
        "Missing prior-week strength columns: "
        + ", ".join(missing_prior_columns)
    )

prior_strength = prior_strength[
    required_prior_columns
].rename(
    columns={
        "weekly_team_strength": "entering_team_strength",
        "weekly_strength_rank": "entering_strength_rank"
    }
)

print(
    f"Week {PRIOR_WEEK} entering ratings loaded:",
    len(prior_strength)
)

display(
    prior_strength.sort_values(
        "entering_strength_rank"
    )
)

Week 2 entering ratings loaded: 32


,team,entering_team_strength,entering_strength_rank
0,BUF,5.233180,1
1,SEA,4.613421,2
2,BAL,4.574111,3
3,DET,4.101898,4
4,SF,4.033455,5
5,JAX,3.713136,6
6,LA,3.095785,7
7,KC,3.052244,8
8,PHI,2.618585,9
9,HOU,2.393466,10


In [7]:
team_games = team_games.merge(
    prior_strength[
        [
            "team",
            "entering_team_strength",
            "entering_strength_rank"
        ]
    ],
    on="team",
    how="left",
    validate="many_to_one"
)

opponent_strength = prior_strength[
    [
        "team",
        "entering_team_strength",
        "entering_strength_rank"
    ]
].rename(
    columns={
        "team": "opponent",
        "entering_team_strength": "opponent_entering_strength",
        "entering_strength_rank": "opponent_entering_rank"
    }
)

team_games = team_games.merge(
    opponent_strength,
    on="opponent",
    how="left",
    validate="many_to_one"
)

display(
    team_games[
        [
            "team",
            "entering_strength_rank",
            "entering_team_strength",
            "opponent",
            "opponent_entering_rank",
            "opponent_entering_strength",
            "points_for",
            "points_against",
            "point_margin"
        ]
    ].sort_values(
        "entering_strength_rank"
    )
)

,team,entering_strength_rank,entering_team_strength,opponent,opponent_entering_rank,opponent_entering_strength,points_for,points_against,point_margin
3,BUF,1,5.233180,DET,4,4.101898,41.0,31.0,10.0
27,SEA,2,4.613421,ARI,25,-2.678016,31.0,7.0,24.0
2,BAL,3,4.574111,NO,22,-1.639387,17.0,24.0,-7.0
10,DET,4,4.101898,BUF,1,5.233180,31.0,41.0,-10.0
28,SF,5,4.033455,MIA,27,-3.609921,35.0,13.0,22.0
14,JAX,6,3.713136,DEN,12,1.819507,13.0,20.0,-7.0
16,LA,7,3.095785,NYG,26,-2.789195,28.0,6.0,22.0
15,KC,8,3.052244,IND,20,-0.967774,33.0,30.0,3.0
25,PHI,9,2.618585,TEN,32,-7.830677,24.0,20.0,4.0
12,HOU,10,2.393466,CIN,19,-0.239434,6.0,20.0,-14.0


# Expected Margin and Opponent-Adjusted Performance

The weekly update is based on how a team performed relative to expectation.

For each team game:

`performance residual = actual margin - expected margin`

Expected margin uses the difference between the two teams' entering strength ratings plus home field advantage.

This means beating a strong opponent is more informative than producing the same result against a weak opponent.

In [8]:
HOME_FIELD_POINTS = 1.5

team_games["expected_margin"] = (
    team_games["entering_team_strength"]
    - team_games["opponent_entering_strength"]
    + np.where(
        team_games["is_home"] == 1,
        HOME_FIELD_POINTS,
        -HOME_FIELD_POINTS
    )
)

team_games["performance_residual"] = (
    team_games["point_margin"]
    - team_games["expected_margin"]
)

display(
    team_games[
        [
            "team",
            "opponent",
            "entering_team_strength",
            "opponent_entering_strength",
            "point_margin",
            "expected_margin",
            "performance_residual"
        ]
    ].sort_values(
        "performance_residual",
        ascending=False
    )
)

,team,opponent,entering_team_strength,opponent_entering_strength,point_margin,expected_margin,performance_residual
4,CAR,ATL,-6.366575,-1.914606,31.0,-5.951970,36.951970
27,SEA,ARI,4.613421,-2.678016,24.0,5.791437,18.208563
6,CIN,HOU,-0.239434,2.393466,14.0,-4.132900,18.132900
18,LV,LAC,-4.526676,-0.089319,12.0,-5.937356,17.937356
21,NE,PIT,1.660188,1.127236,17.0,2.032952,14.967048
22,NO,BAL,-1.639387,4.574111,7.0,-7.713498,14.713498
16,LA,NYG,3.095785,-2.789195,22.0,7.384980,14.615020
8,DAL,WAS,-1.053063,-2.168535,17.0,2.615472,14.384528
28,SF,MIA,4.033455,-3.609921,22.0,9.143376,12.856624
7,CLE,TB,-6.150395,0.101026,4.0,-7.751421,11.751421


# Control Extreme Single-Game Results

NFL game margins contain substantial single-game noise.

A blowout should affect team strength, but one extreme result should not completely redefine the team's rating.

Performance residuals are therefore capped before the weekly learning rate is applied.

In [9]:
RESIDUAL_SCALE = 14.0
MAX_WEEKLY_UPDATE = 1.5

team_games["weekly_strength_update"] = (
    MAX_WEEKLY_UPDATE
    * np.tanh(
        team_games["performance_residual"]
        / RESIDUAL_SCALE
    )
)

display(
    team_games[
        [
            "team",
            "opponent",
            "point_margin",
            "expected_margin",
            "performance_residual",
            "weekly_strength_update"
        ]
    ].sort_values(
        "weekly_strength_update",
        ascending=False
    )
)

,team,opponent,point_margin,expected_margin,performance_residual,weekly_strength_update
4,CAR,ATL,31.0,-5.951970,36.951970,1.484783
27,SEA,ARI,24.0,5.791437,18.208563,1.292821
6,CIN,HOU,14.0,-4.132900,18.132900,1.290726
18,LV,LAC,12.0,-5.937356,17.937356,1.285222
21,NE,PIT,17.0,2.032952,14.967048,1.183669
22,NO,BAL,7.0,-7.713498,14.713498,1.173271
16,LA,NYG,22.0,7.384980,14.615020,1.169153
8,DAL,WAS,17.0,2.615472,14.384528,1.159335
28,SF,MIA,22.0,9.143376,12.856624,1.087660
7,CLE,TB,4.0,-7.751421,11.751421,1.028227


# Offensive and Defensive Weekly Signals

Overall team strength uses the opponent adjusted margin residual.

We also preserve separate offensive and defensive signals because the weekly game projection model needs scoring information.

These are weekly updates, not cumulative season averages.

Opponent entering strength is used conservatively as context until separate entering offensive and defensive strength ratings are maintained.

In [10]:
league_points_per_team_game = (
    team_games["points_for"].mean()
)

OPPONENT_CONTEXT_RATE = 0.25
OFFENSE_DEFENSE_UPDATE_RATE = 0.15

team_games["raw_offense_performance"] = (
    team_games["points_for"]
    - league_points_per_team_game
)

team_games["raw_defense_performance"] = (
    league_points_per_team_game
    - team_games["points_against"]
)

team_games["adjusted_offense_performance"] = (
    team_games["raw_offense_performance"]
    + (
        OPPONENT_CONTEXT_RATE
        * team_games["opponent_entering_strength"]
    )
)

team_games["adjusted_defense_performance"] = (
    team_games["raw_defense_performance"]
    + (
        OPPONENT_CONTEXT_RATE
        * team_games["opponent_entering_strength"]
    )
)

team_games["weekly_offense_adjustment"] = (
    team_games["adjusted_offense_performance"]
    * OFFENSE_DEFENSE_UPDATE_RATE
)

team_games["weekly_defense_adjustment"] = (
    team_games["adjusted_defense_performance"]
    * OFFENSE_DEFENSE_UPDATE_RATE
)

display(
    team_games[
        [
            "team",
            "opponent",
            "points_for",
            "points_against",
            "opponent_entering_strength",
            "raw_offense_performance",
            "adjusted_offense_performance",
            "weekly_offense_adjustment",
            "raw_defense_performance",
            "adjusted_defense_performance",
            "weekly_defense_adjustment"
        ]
    ].sort_values(
        "team"
    )
)

,team,opponent,points_for,points_against,opponent_entering_strength,raw_offense_performance,adjusted_offense_performance,weekly_offense_adjustment,raw_defense_performance,adjusted_defense_performance,weekly_defense_adjustment
0,ARI,SEA,7.0,31.0,4.613421,-13.21875,-12.065395,-1.809809,-10.78125,-9.627895,-1.444184
1,ATL,CAR,3.0,34.0,-6.366575,-17.21875,-18.810394,-2.821559,-13.78125,-15.372894,-2.305934
2,BAL,NO,17.0,24.0,-1.639387,-3.21875,-3.628597,-0.544290,-3.78125,-4.191097,-0.628665
3,BUF,DET,41.0,31.0,4.101898,20.78125,21.806725,3.271009,-10.78125,-9.755775,-1.463366
4,CAR,ATL,34.0,3.0,-1.914606,13.78125,13.302599,1.995390,17.21875,16.740099,2.511015
5,CHI,MIN,3.0,9.0,1.891707,-17.21875,-16.745823,-2.511873,11.21875,11.691677,1.753752
6,CIN,HOU,20.0,6.0,2.393466,-0.21875,0.379617,0.056942,14.21875,14.817117,2.222567
7,CLE,TB,23.0,19.0,0.101026,2.78125,2.806507,0.420976,1.21875,1.244007,0.186601
8,DAL,WAS,37.0,20.0,-2.168535,16.78125,16.239116,2.435867,0.21875,-0.323384,-0.048508
9,DEN,JAX,20.0,13.0,3.713136,-0.21875,0.709534,0.106430,7.21875,8.147034,1.222055


# Build Week-Level Feature Table

The final feature table contains only the new evidence that should move the prior week's ratings.

For Week 3:

`Week 3 strength = Week 2 strength + Week 2 performance update`

The previous week's rating therefore acts as the prior, and older games are not counted again.

In [11]:
team_features = (
    team_games
    .groupby("team", as_index=False)
    .agg(
        games_in_update=("game_id", "count"),

        entering_team_strength=(
            "entering_team_strength",
            "first"
        ),

        entering_strength_rank=(
            "entering_strength_rank",
            "first"
        ),

        average_opponent_strength=(
            "opponent_entering_strength",
            "mean"
        ),

        average_opponent_rank=(
            "opponent_entering_rank",
            "mean"
        ),

        actual_margin=(
            "point_margin",
            "mean"
        ),

        expected_margin=(
            "expected_margin",
            "mean"
        ),

        performance_residual=(
            "performance_residual",
            "mean"
        ),

        weekly_strength_update=(
            "weekly_strength_update",
            "mean"
        ),

        weekly_offense_adjustment=(
            "weekly_offense_adjustment",
            "mean"
        ),

        weekly_defense_adjustment=(
            "weekly_defense_adjustment",
            "mean"
        )
    )
)

display(
    team_features[
        [
            "team",
            "entering_strength_rank",
            "entering_team_strength",
            "average_opponent_rank",
            "average_opponent_strength",
            "actual_margin",
            "expected_margin",
            "performance_residual",
            "weekly_strength_update",
            "weekly_offense_adjustment",
            "weekly_defense_adjustment"
        ]
    ].sort_values(
        "weekly_strength_update",
        ascending=False
    )
)

,team,entering_strength_rank,entering_team_strength,average_opponent_rank,average_opponent_strength,actual_margin,expected_margin,performance_residual,weekly_strength_update,weekly_offense_adjustment,weekly_defense_adjustment
4,CAR,31,-6.366575,23.0,-1.914606,31.0,-5.951970,36.951970,1.484783,1.995390,2.511015
27,SEA,2,4.613421,25.0,-2.678016,24.0,5.791437,18.208563,1.292821,1.516762,1.882387
6,CIN,19,-0.239434,10.0,2.393466,14.0,-4.132900,18.132900,1.290726,0.056942,2.222567
18,LV,28,-4.526676,18.0,-0.089319,12.0,-5.937356,17.937356,1.285222,0.863838,0.929463
21,NE,14,1.660188,15.0,1.127236,17.0,2.032952,14.967048,1.183669,0.009459,2.625084
22,NO,22,-1.639387,3.0,4.574111,7.0,-7.713498,14.713498,1.173271,0.738717,0.654342
16,LA,7,3.095785,26.0,-2.789195,22.0,7.384980,14.615020,1.169153,1.062593,2.028218
8,DAL,21,-1.053063,24.0,-2.168535,17.0,2.615472,14.384528,1.159335,2.435867,-0.048508
28,SF,5,4.033455,27.0,-3.609921,22.0,9.143376,12.856624,1.087660,2.081815,0.947440
7,CLE,30,-6.150395,17.0,0.101026,4.0,-7.751421,11.751421,1.028227,0.420976,0.186601


# Sanity Checks

Before saving, verify that:

- all 32 teams are represented
- every team has an entering Week 2 rating
- every opponent has an entering Week 2 rating
- no Week 3 game results were used
- weekly strength updates remain within the intended cap

In [12]:
print("Teams in feature table:", len(team_features))
print(
    "Unique teams:",
    team_features["team"].nunique()
)

print(
    "Missing entering team strengths:",
    team_features[
        "entering_team_strength"
    ].isna().sum()
)

print(
    "Missing opponent strengths:",
    team_features[
        "average_opponent_strength"
    ].isna().sum()
)

print(
    "Largest positive weekly update:",
    round(
        team_features[
            "weekly_strength_update"
        ].max(),
        3
    )
)

print(
    "Largest negative weekly update:",
    round(
        team_features[
            "weekly_strength_update"
        ].min(),
        3
    )
)

if (
    completed_games["week"] >= TARGET_WEEK
).any():
    raise ValueError(
        "DATA LEAKAGE: target-week results "
        "were found in completed games."
    )

if len(team_features) != 32:
    print()
    print(
        "WARNING: Feature table does not "
        "contain all 32 teams."
    )

Teams in feature table: 32
Unique teams: 32
Missing entering team strengths: 0
Missing opponent strengths: 0
Largest positive weekly update: 1.485
Largest negative weekly update: -1.485


# Save In Season Feature Table

This output will be consumed by:

`04_Weekly_Team_Strength.ipynb`

The file contains only the new in season evidence. It does not overwrite or replace the preseason team strength table.


In [13]:
output_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_inseason_features.parquet"
)

team_features = team_features.sort_values(
    "team"
).reset_index(drop=True)

team_features.to_parquet(
    output_path,
    index=False
)

print("Saved:", output_path)


Saved: ..\..\data\processed\weekly\week_03_inseason_features.parquet


# Interpretation

The weekly model is now sequential.

For Week 3:

1. Week 2 team strength is treated as the entering prior.
2. Only Week 2 game results provide new evidence.
3. Performance is evaluated relative to the opponent's entering strength.
4. Extreme single game residuals are capped.
5. A conservative learning rate determines the weekly rating movement.
6. The resulting update will be added to the Week 2 rating in `04_Weekly_Team_Strength.ipynb`.

This prevents older games from being counted repeatedly and makes strength of opponent part of every weekly update.

The frozen preseason model remains unchanged.